## Fetch and Display KRX Stock Data
KRX, KOSDAQ, KOSPI, 여러 종목

In [1]:
# _tsa_00.py
# time series stuff, basic
# 그리기
# 시계열, 추세, 계절성 조정, 이동평균, 이런 것들.
# 자기공분산, 자기상관, 단위근
# ARIMA 중 AR(p)
# 기타... 단순한 내용만 간략하게.

# 코랩 추가.
!pip install yfinance
#!pip install pykrx   # 이건 스크래핑, 무단 자료조회 모듈임. 스킵.
!pip install finance-datareader

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf                       ## NYSE 데이터 모듈
import FinanceDataReader as fdr             ## KOSPI, NYSE
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭.
from statsmodels.tsa.ar_model import AutoReg


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.6 MB/s eta 0:00:00


### 종목 코드

In [9]:
# 별도로 종목 리스트 받음.
# 20260901_krx_list 1. 라이브러리 설치 (코랩의 경우 맨 앞에 ! 추가)
#!pip install yfinance
# import yfinance as yf
krx_list_url = 'https://github.com/bahn28/salad/blob/main/kospi_stock_list_20260901.csv?raw=true'
df_dat_u = pd.read_csv(krx_list_url)
# 2. KOSPI 상장 종목 전체 가져오기
#df_kospi = fdr.StockListing('KOSPI')  # 불안정, 이것보다는 공공데이터포털 API 사용.
#print(df_kospi[['Code', 'Name', 'Sector']].head(7))
# 0  005930  KR7005930003    삼성전자  KOSPI  NaN   257000          2   -24500
# 1  000660  KR7000660001  SK하이닉스  KOSPI  NaN  1688000          2   -42000
# 2  005935  KR7005931001   삼성전자우  KOSPI  NaN   190500          2   -16500
# 3  402340  KR7402340004   SK스퀘어  KOSPI  NaN  1072000          2   -51000
# 4  009150  KR7009150004    삼성전기  KOSPI  NaN

df_dat_u.head() # 영어 코드로 작성된걸 받자. 검색 재실행.
# 코드, 종목명 짝을 적어놓차.

,basDt,srtnCd,isinCd,itmsNm,mrktCtg,clpr,vs,fltRt,mkp,hipr,lopr,trqu,trPrc,lstgStCnt,mrktTotAmt
0,20260901,900110,HK0000057197,딥커머스,KOSDAQ,1201,0,0.00,0,0,0,0,0,18437131,22142994331
1,20260901,900270,HK0000214814,헝셩그룹,KOSDAQ,3500,175,5.26,3400,4140,3195,3093709,11737502970,5112288,17893008000
2,20260901,900260,HK0000295359,로스웰,KOSDAQ,1695,45,2.73,1650,1814,1643,33876,59558690,46029706,78020351670
3,20260901,900290,HK0000307485,GRT,KOSDAQ,4200,-10,-0.24,4210,4450,4105,455730,1962909454,83510544,350744284800
4,20260901,900300,HK0000312568,오가닉티코스메틱,KOSDAQ,4075,-145,-3.44,4235,4370,3955,618169,2575663828,20580097,83863895275


### 개별 종목, 가격, 거래량

In [3]:

# 1. 삼성전자(005930)의 2025년부터 현재까지의 주가 데이터 수집
tickers = ['005930', '000660']    # 삼성전자 코드, 티커.
df_dat = fdr.DataReader(tickers, '2023-01-01')  # 삼성전자
#ticker_2 = '000660'  # SK하이익스
#df_dat_2 = fdr.DataReader(ticker_2, '2023-01-01')  # 하이닉스

# # 2. 가격(종가) 데이터 읽기
# # 최신 버전 FinanceDataReader의 주가 컬럼명은 'Close'입니다.

#prc_cls = df_dat['Close']
#prc_cls_hy = df_dat_2['Close']

# # 3. 거래량(볼륨) 데이터 읽기
#vlm = df_dat['Volume']
#vlm_2 = df_dat_2['Volume']

# # 4. 상위 5개 데이터 결합해서 눈으로 확인하기
# print(df_dat[['Close', 'Volume']].head())
# print(df_dat.head(15) )
df_dat.info()
df_dat.head()


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 900 entries, 2023-01-02 to 2026-09-09
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   005930  900 non-null    int64
 1   000660  900 non-null    int64
dtypes: int64(2)
memory usage: 21.1 KB


,005930,000660
Date,,
2023-01-02,55500,75700
2023-01-03,55400,75600
2023-01-04,57800,81000
2023-01-05,58200,81400
2023-01-06,59000,83100


In [4]:
# daily stock prices and trading volumes
# 변수로 받는 방법.
yymmdd = df_dat.index   # 자료를 불러올 때, 자동으로 기준 인덱스로 설정된다고 함.
yymmdd = pd.to_datetime( yymmdd )

stck_prc = df_dat['Close']
stck_prc_dff = stck_prc.diff()
stck_prc_pct = stck_prc.pct_change()

stck_vlm = df_dat['Volume']
stck_vlm_dff = stck_vlm.diff()
stck_vlm_pct = stck_vlm.pct_change()

stck_prc_2 = df_dat_2['Close']
stck_prc_dff_2 = stck_prc_2.diff()
stck_prc_pct_2 = stck_prc_2.pct_change()

stck_vlm_2 = df_dat_2['Volume']
stck_vlm_dff_2 = stck_vlm_2.diff()
stck_vlm_pct_2 = stck_vlm_2.pct_change()


KeyError: 'Close'

### plots, daily price level

In [ ]:

plt.figure(figsize=(8, 5))
#plt.subplot(figsize=(8,5  ))
plt.plot(yymmdd, stck_prc, color='blue', linewidth=1)
plt.title(f" Closing Prices of {ticker} ")

plt.figure(figsize=(8, 5))
#plt.subplot(figsize=(8,5  ))
plt.plot(yymmdd, stck_prc_2, color='blue', linewidth=1)
plt.title(f" Closing Prices of {ticker_2} ")


### plots, daily price/volume changes

In [ ]:

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
#axes[0].plot(yymmdd, stck_prc, color='blue', linewidth=0.5)
axes[0].plot(yymmdd, stck_prc_pct, color='blue', linewidth=0.5)
axes[1].plot(yymmdd, stck_vlm, color='red', linewidth=0.5)
# axes[0].set_title(f" Closing Prices of {ticker} ")
axes[0].set_title(f" Daily Price Changes of {ticker} ")
axes[1].set_title(f" Daily Trading Volumes of {ticker} ")
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
#axes[0].plot(yymmdd, stck_prc, color='blue', linewidth=0.5)
axes[0].plot(yymmdd, stck_prc_pct_2, color='blue', linewidth=0.5)
axes[1].plot(yymmdd, stck_vlm_2, color='red', linewidth=0.5)
# axes[0].set_title(f" Closing Prices of {ticker} ")
axes[0].set_title(f" Daily Price Changes of {ticker_2} ")
axes[1].set_title(f" Daily Trading Volumes of {ticker_2} ")
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=6))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%y/%m'))


### histogram, pct changes

In [ ]:

plt.figure(figsize=(8, 5))
plt.hist(stck_prc_pct,
         color='blue',
         linewidth=1,
         bins = 16,
         facecolor='None',
         edgecolor='black',
         density=True
         )
plt.title(f" distribution of daily percentage change of {ticker} ")
#plt.show()

plt.hist(stck_prc_pct_2,
         color='blue',
         linewidth=1,
         bins = 16,
         facecolor='None',
         edgecolor='blue',
         density=True
         )
plt.title(f" distribution of daily percentage change of {ticker_2} ")
#plt.show()